## Apartado 6. Detección de contenido inapropiado usando ZSL, FSL y Chain-of-thought

In [1]:
import zstandard as zstd
import json
import argparse
import io
import sys
from pathlib import Path
from datetime import datetime, UTC
import random
import os
import pandas as pd
import numpy as np

In [2]:
submissions_file = "RS_OpinionesPolemicas_2025.zst"
comments_file = "RC_OpinionesPolemicas_2025.zst"

### Compilación del corpus de Opiniones Polémicas

In [31]:
#!/usr/bin/env python3
"""
=====================
Este ejemplo procesa archivos de un volcado de Reddit en formato .zst para extraer
submissions y sus comentarios de un subreddit específico.
"""

def stream_zst_file(filepath):
    """
    Genera objetos JSON línea por línea desde un archivo .zst
    """
    
    dctx = zstd.ZstdDecompressor(max_window_size=2**31)

    with open(filepath, 'rb') as fh:
        with dctx.stream_reader(fh) as reader:
            text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='ignore')

            for line in text_stream:
                line = line.strip()
                if not line:
                    continue
                try:
                    yield json.loads(line)
                except json.JSONDecodeError:
                    continue


def extract_submissions(filepath, subreddit, n_submissions=80, min_comments=250):
    """
    Extrae N submissions de un subreddit con al menos min_comments comentarios.
    Captura los atributos más importantes del dump.
    """
    random.seed(42) # Esta vez si establecemos una semilla a la hora de compilar el corpus para que sea reproducible
    submissions = []
    subreddit_lower = subreddit.lower()

    print(f"🔍 Buscando submissions en el subreddit {subreddit} con ≥ {min_comments} comentarios...")

    count = 0
    scanned = 0

    for obj in stream_zst_file(filepath):
        if obj.get('subreddit', '').lower() != subreddit_lower:
            continue

        scanned += 1
        num_comments = obj.get('num_comments', 0)

        if num_comments < min_comments:
            continue

        # Copiar los atributos más importantes
        submission = {
            "id": obj.get("id"),
            "subreddit": obj.get("subreddit"),
            "author": obj.get("author"),
            "title": obj.get("title"),
            "selftext": obj.get("selftext", ""),
            "url": obj.get("url", ""),
            "score": obj.get("score", 0),
            "num_comments": obj.get("num_comments", 0),
            "created_utc": obj.get("created_utc"),
            "is_self": obj.get("is_self", ""),
            "link_flair_text": obj.get("link_flair_text")
        }

        # Añadir campos calculados
        submission['name'] = f"t3_{obj.get('id')}"
        submission['created_datetime'] = datetime.fromtimestamp(
            obj.get('created_utc', 0), UTC
        ).isoformat() if obj.get('created_utc') else None
        submission['comments'] = []  # Se llenará después con los comentarios

        """
        Ahora hacemos la distribución en el tiempo.
        Cuando se ha llenado la lista con los N submissions se calcula un número al azar entre 0 y el número de hilos que llevamos procesados.
        Si este número está entre 0 y N-submissions se cambia el submission correspondiente a ese número por el nuevo.
        De esta forma nos aseguramos tener hilos de toda la distribución temporal.
        Si un subreddit tiene 100.000 hilos, cuando lleve procesados la mitad, count valdrá 50.000, y la probabilidad de que se añada ese
        hilo  será N-submissions/50.000. Esta lógica para extraer los hilos/comentarios a lo largo del tiempo se llama Reservoir Sampling
        """

        if count < n_submissions:
            submissions.append(submission)
        else:
            j = random.randint(0, count)
            if j < n_submissions:
                submissions[j] = submission

        count += 1

    print(f"\n  Escaneadas: {scanned} del subreddit")
    print(f"  Encontradas: {len(submissions)} con ≥{min_comments} comentarios\n")
    return submissions


def extract_comments_for_submissions(filepath, submissions, num_comments=50):
    """
    Extrae N comentarios para cada submission.
    Captura los atributos más importantes del dump.
    """
    random.seed(42) # Esta vez si establecemos una semilla a la hora de compilar el corpus para que sea reproducible
    if num_comments <= 0:
        print("⏭️  num_comments=0, saltando búsqueda de comentarios")
        return

    if not submissions:
        print("⏭️  Sin submissions")
        return

    submission_map = {
    subreddit: {s['name']: s for s in lista_hilos}
    for subreddit, lista_hilos in submissions.items()
    }
    comment_count = {
    subreddit: {s['name']: 0 for s in lista_hilos}
    for subreddit, lista_hilos in submissions.items()
}

    pending = set(name for subdiccionario in submission_map.values()for name in subdiccionario)

    print(f"🔍 Buscando hasta {num_comments} comentarios por submission...")

    total_comments = 0

    for obj in stream_zst_file(filepath):
        link_id = obj.get('link_id')
        if link_id not in pending:
            continue

        # Copiamos los atributos más importantes
        comment = {
            "id": obj.get("id"),
            "link_id": obj.get("link_id"),
            "parent_id": obj.get("parent_id"),
            "subreddit": obj.get("subreddit"),
            "author": obj.get("author"),
            "body": obj.get("body", ""),
            "score": obj.get("score", 0),
            "created_utc": obj.get("created_utc"),
            "is_submitter": obj.get("is_submitter", False)
        }
        
        
        # Pasamos el nombre del subreddit a minúsculas
        subreddit = obj.get("subreddit").lower()

        # Añadimos campos calculados
        comment['name'] = f"t1_{obj.get('id')}"
        comment['created_datetime'] = datetime.fromtimestamp(
            obj.get('created_utc', 0), UTC
        ).isoformat() if obj.get('created_utc') else None
        

        # Hacemos la lógica 'Reservoir Sampling' para obtener una lista de comentarios distribuidos en el tiempo
        current_comments_list = submission_map[subreddit][link_id]['comments']
        
        # Si hay menos de 'num_comments' comentarios, se añade el comentario directamente
        if comment_count[subreddit][link_id] < num_comments:
            current_comments_list.append(comment)
            
        # Si ya hay 'num_comments' comentarios guardados, se aplica la lógica de 'Reservoir Sampling' para conseguir que los 
        # comentarios finales estén espaciados en el tiempo
        else:
            j = random.randint(0, comment_count[subreddit][link_id])
            if j < num_comments:
                current_comments_list[j] = comment
        
        # Actualizamos los contadores de comentarios encontrados para esa submissions y el contador total
        comment_count[subreddit][link_id] += 1
        total_comments += 1

    print(f"  Total: {total_comments} comentarios encontrados\n")
    for subreddit in submissions:
        print(f"Subrredit: {subreddit}")
        for s in submissions[subreddit]:
            print(f"  📝 \"{s.get('title', '')[:50]}...\" → {len(s['comments'])} comentarios")

In [32]:
# Aunque solo tengamos un "subreddit" esta vez, extraeremos los hilos de la misma forma que en el apartado 1.
# Esto lo hacemos así para no tener que cambiar la función de extraer comentarios.
subreddits = ["opinionespolemicas"]
num_submissions = 10

submissions = {subreddit: _ for subreddit in subreddits}
print(f"{'='*60}")
print(f"Submissions file: {submissions_file}")
print(f"Comments file: {comments_file}")
print(f"Número de submissions (hilos o posts): {num_submissions}")
print(f"{'='*60}\n")

# 1. Extraemos las submissions 
for subreddit in subreddits:
  submissions[subreddit] = extract_submissions(
    submissions_file,
    subreddit,
    num_submissions
  )

  if not submissions[subreddit]:
    print(f"❌ No se encontraron submissions")
    sys.exit(1)

Submissions file: RS_OpinionesPolemicas_2025.zst
Comments file: RC_OpinionesPolemicas_2025.zst
Número de submissions (hilos o posts): 10

🔍 Buscando submissions en el subreddit opinionespolemicas con ≥ 250 comentarios...

  Escaneadas: 17854 del subreddit
  Encontradas: 10 con ≥250 comentarios



In [33]:
# 2. Extraer comentarios
extract_comments_for_submissions(comments_file, submissions, 50)

🔍 Buscando hasta 50 comentarios por submission...
  Total: 4969 comentarios encontrados

Subrredit: opinionespolemicas
  📝 "Si no creen en el transgenerismo, entonces cual es..." → 50 comentarios
  📝 "Los españoles y latinos deberían dejar de odiarse ..." → 50 comentarios
  📝 "Por qué hay mujeres que se creen expertas en el se..." → 50 comentarios
  📝 "Gratis debería ser la quimioterapia en vez del Abo..." → 50 comentarios
  📝 "No existe una raza/etnia superior..." → 50 comentarios
  📝 "Toda la inclusión es forzada...." → 50 comentarios
  📝 "Los trumpistas latinoamericanos son gente frustrad..." → 50 comentarios
  📝 "No existe ninguna buena razón para creer en lo sob..." → 50 comentarios
  📝 "No entiendo a los pro-vida..." → 50 comentarios
  📝 "Si el hombre no te gusta no le aceptes comida muer..." → 50 comentarios


In [6]:
# 3. Creamos el resultado final
for subreddit in subreddits:
  result = {
    'subreddit': subreddit,
    'extraction_date': datetime.utcnow().isoformat(),
    'num_submissions': len(submissions[subreddit]),
    'total_comments': sum(len(s['comments']) for s in submissions[subreddit]),
    'submissions': submissions[subreddit]
  }

  # 4. Mostramos los JSON
  print(f"   - {result['num_submissions']} submissions (hilos o posts)")
  print(f"   - {result['total_comments']} comentarios totales")

  output= subreddit+".json"
  
  # 5. Guardamos los JSON
  output_path = Path(output)
  with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

   - 10 submissions (hilos o posts)
   - 500 comentarios totales


In [ ]:
from huggingface_hub import login
mi_token = "aqui va el token de hf"
login(token=mi_token)

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")
# Cargamos el modelo en GPU si la hay poniendo el device_map en "auto"
model = AutoModelForCausalLM.from_pretrained("google/gemma-3-1b-it", device_map="auto")


C:\Users\30mmn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Definimos una función para obtener las respuestas del SLM y realizar un post-procesado para quedarnos solo con la etiqueta de si el texto contiene contenido apropiado o no

In [9]:
import re
def generate_response(base_prompt, text):
    # Estructuramos los mensajes de entrada en el formato requerido por Gemma
    messages = [
        {
            "role": "user",
            "content": base_prompt + " " + text,
        },
    ]
    # Aplicamos un template de chat al mensaje de entrada utilizando el tokenizador.
    # Esto formatea los mensajes según el formato esperado por el modelo, añadiendo un prompt
    inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    ).to(model.device)

    output = model.generate(**inputs, max_new_tokens=10)
    output = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    # Vamos a realizar un posprocesado para quedarnos solo con la etiqueta
    try:
        # Intentamos obtener la etiqueta que debería venir entre corchetes
        regex = re.compile(r'\[(.+)\]')
        response = regex.search(output).group(1)
    except:
        # Si la salida generada no sigue el formato esperado, devolvemos la salida completa
        response = output
    
    return response

Primero vamos a obtener las respuestas con las etiquetas usando la estrategia Zero-Shot Learning (ZSL). Además, crearemos un fichero json nuevo donde guardar esta nueva información.

In [ ]:
zero_shot_prompt = """
Eres un modelo cuya tarea es determinar hay o no hay contenido inapropiado en el texto recibido.
Considero contenido inapropiado lo siguiente: 
- Que haya insultos o lenguaje soez.
- Que haya discurso de odio (homofobia, racismo, misoginia).
- Que haya violencia verbal o amenazas.
Si el texto tiene algo de lo mencionado, clasificalo como inapropiado, de lo contrario como apropiado.
Formato de salida: [clasificación].
Por favor, determina si el siguiente texto es inapropiado o apropiado:"""

# Extracción de los textos
filepath = Path(f"opinionespolemicas.json")
output_path = Path(f"opinionespolemicas_icl.json")
with open(filepath, "r", encoding='utf-8') as f:
    data = json.load(f)

for submission in data["submissions"]:
    for comment in submission["comments"]:
        texto = comment.get("body", "")
        
        # Obtenemos la etiqueta
        response = generate_response(zero_shot_prompt, texto)
        # Añadimos un campo al comentario con la etiqueta de ZSL
        comment['ZSL'] = response

# Guardamos la nueva información en un nuevo json     
with open(output_path, "w", encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

Ahora emplearemos la estrategia Few-Shot Learning (FSL) para obtener las respuestas y sobreescribiremos el archivo anterior para añadir esta información.

In [ ]:
few_shot_prompt = """
Eres un modelo cuya tarea es determinar hay o no hay contenido inapropiado en el texto recibido.
Considero contenido inapropiado lo siguiente: 
- Que haya insultos o lenguaje soez.
- Que haya discurso de odio (homofobia, racismo, misoginia).
- Que haya violencia verbal o amenazas.
Si el texto tiene algo de lo mencionado, clasificalo como inapropiado, de lo contrario como apropiado.
Ejemplos:
Texto: Me cago en todos los putos políticos de mierda de este país
Clasificación: [Inapropiado]
Texto: Me han dado la nota del examen y he sacado un 9. ¡ Estoy muy felizzz!
Clasificación: [Apropiado]
Texto: Las mujeres deberían estar preparando la cena, no jugando al fútbol.
Clasificación: [Inapropiado]
Texto: Ojalá todos los inmigrantes se fueran a su puto país de mierda
Clasificación: [Inapropiado]
Texto: La sanidad pública va muy mal, tienes que esperar 4 meses para conseguir cita.
Clasificación: [Apropiado]
Texto: Todos los de esta red social sois unos chupapollas que no teneis opinión crítica con nada.
Clasificación: [Inapropiado]
Texto: Los jóvenes de ahora estáis todos amariconaos.
Clasificación: [Inapropiado]
Texto: y por qué tenemos que estar encerrados sin poder salir?
Clasificación: [Apropiado]
Texto: Lo que has escrito no tiene relevancia alguna con el tema que estamos tratando.
Clasificación: [Apropiado]
Texto: te faltan un par de veranos? lo que has escrito es una soberana gilipollez
Clasificación: [Inapropiado]


Formato de salida: [clasificación].
Por favor, determina si el siguiente texto es inapropiado o apropiado:"""

# Extracción de los textos
filepath = Path(f"opinionespolemicas_icl.json")
with open(filepath, "r", encoding='utf-8') as f:
    data = json.load(f)

for submission in data["submissions"]:
    for comment in submission["comments"]:
        texto = comment.get("body", "")
        
        # Obtenemos la etiqueta
        response = generate_response(few_shot_prompt, texto)
        # Añadimos un campo al comentario con la etiqueta de FSL
        comment['FSL'] = response

# Guardamos la nueva información en el mismo json     
with open(filepath, "w", encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

Por último, usaremos la estrategia Chain-of-Thought (CoT) para obtener las etiquetas de los comentarios. Para esta estrategia, primero crearemos una nueva función para generar la respuesta, a la que le pasaremos también el número de tokens máximos generados a la salida. En principio usaremos 200 tokens como máximo a la salida para que no se corte la respuesta del modelo, ya que nos devolverá el razonamiento que ha seguido para etiquetar cada comentario; aunque como antes, nos quedaremos solo con la etiqueta. Aunque solo nos quedemos con la etiqueta, le pasamos los 200 tokens como máximo a la salida, para que no se corte la respuesta, ya que la etiqueta será lo último que genere el modelo. 

In [12]:
def generate_response_cot(base_prompt, text, max_new_tokens):
    # Estructuramos los mensajes de entrada en el formato requerido por Gemma
    messages = [
        {
            "role": "user",
            "content": base_prompt + " " + text,
        },
    ]
    # Aplicamos un template de chat al mensaje de entrada utilizando el tokenizador.
    # Esto formatea los mensajes según el formato esperado por el modelo, añadiendo un prompt
    inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    ).to(model.device)

    output = model.generate(**inputs, max_new_tokens=max_new_tokens)
    output = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    # Vamos a realizar un posprocesado para quedarnos solo con la etiqueta
    try:
        # Intentamos obtener la etiqueta que debería venir entre corchetes
        regex = re.compile(r'\[(.+)\]')
        response = regex.search(output).group(1)
    except:
        # Si la salida generada no sigue el formato esperado, devolvemos la salida completa
        response = output
    
    return response

Para CoT, le pasaremos menos ejemplos en el prompt al modelo, ya que estos van junto con el razonamiento, para que el prompt no sea demasiado largo.

In [13]:
cot_prompt = """
Eres un modelo cuya tarea es determinar hay o no hay contenido inapropiado en el texto recibido.
Considero contenido inapropiado lo siguiente: 
- Que haya insultos o lenguaje soez.
- Que haya discurso de odio (homofobia, racismo, misoginia).
- Que haya violencia verbal o amenazas.

Antes de dar tu clasificación final, debes explicar tu razonamiento paso a paso. Para ello:
Análisis: Identifica si en el texto existen palabras malsonantes, insultos, prejuicios discriminatorios o intenciones violentas.
Evaluación: Compara lo encontrado con los tres criterios de contenido inapropiado mencionados arriba y justifica si infringe alguna regla.
Conclusión: Basándote en tu evaluación, decide el veredicto.
Escribe tu proceso mental indicando 'Razonamiento:' y luego, en una nueva línea, da la clasificación final. 
Si el texto tiene algo de lo mencionado, clasificalo como inapropiado, de lo contrario como apropiado.
Ejemplos:
Texto: Me cago en todos los putos políticos de mierda de este país
Razonamiento: El texto contiene un alto nivel de lenguaje soez y vulgar ("putos", "de mierda", "me cago en") dirigido de forma despectiva a un grupo de personas. Esto infringe directamente la regla sobre el uso de insultos o lenguaje soez.
Clasificación: [Inapropiado]

Texto: Las mujeres deberían estar preparando la cena, no jugando al fútbol.
Razonamiento: El texto utiliza un estereotipo machista que relega a la mujer a tareas domésticas e invalida su participación en los deportes. Esto constituye un discurso de odio (misoginia). Por lo tanto, incumple las normas.
Clasificación: [Inapropiado]

Texto: La sanidad pública va muy mal, tienes que esperar 4 meses para conseguir cita.
Razonamiento: El texto es una queja sobre un servicio público. Aunque expresa descontento, no contiene lenguaje soez, ni discurso de odio, ni amenazas. Es una opinión válida.
Clasificación: [Apropiado]

Texto: Lo que has escrito no tiene relevancia alguna con el tema que estamos tratando.
Razonamiento: El texto es una crítica directa y tajante, pero está formulada con un lenguaje neutral y respetuoso. No contiene insultos, amenazas ni ningún tipo de discurso de odio. Es simplemente una corrección en el contexto de un debate.
Clasificación: [Apropiado]


Por favor, determina si el siguiente texto es inapropiado o apropiado:"""

# Extracción de los textos
filepath = Path(f"opinionespolemicas_icl.json")
with open(filepath, "r", encoding='utf-8') as f:
    data = json.load(f)

for submission in data["submissions"]:
    for comment in submission["comments"]:
        texto = comment.get("body", "")
        
        # Obtenemos la etiqueta
        response = generate_response_cot(few_shot_prompt, texto, 200)
        # Añadimos un campo al comentario con la etiqueta de FSL
        comment['CoT'] = response

# Guardamos la nueva información en el mismo json     
with open(filepath, "w", encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

### Análisis de las etiquetas de 10 comentarios

Ahora vamos a seleccionar 10 comentarios y ver las etiquetas que se han generado en cada estrategia para realizar un análisis de estas y poder ver posibles limitaciones de cada una.

In [36]:
filepath = Path(f"opinionespolemicas_icl.json")
with open(filepath, "r", encoding='utf-8') as f:
    data = json.load(f)
comentarios_elegidos = []
count = 0
# Hacemos uso de la lógica Reservoir Sampling para sacar los 10 comentarios
for submission in data["submissions"]:
    for comment in submission["comments"]:
        texto = comment.get("body", "")
        etiquetas = {"ZSL": comment.get("ZSL", ""), "FSL": comment.get("FSL", ""), "CoT": comment.get("CoT", "")}
        if count < 10:
            comentarios_elegidos.append((texto, etiquetas))
        else:
            j = random.randint(0, count)
            if j < 10:
                comentarios_elegidos[j] = (texto, etiquetas)

        count += 1

print(f"{'-'*60}") 
for i, comentario in enumerate(comentarios_elegidos):
    texto, etiquetas = comentario
    print(f"Comentario {i}:")
    print(f"Texto: {texto}\nEtiquetas: {etiquetas}")
    print(f"{'-'*60}")

------------------------------------------------------------
Comentario 0:
Texto: Recuerden que si por los hombres fuera, ya no existiría ninguna minoría cual sea que está sea....
Etiquetas: {'ZSL': 'Por favor, proporciona el texto que deseas que anal', 'FSL': 'Inapropiado', 'CoT': 'Inapropiado'}
------------------------------------------------------------
Comentario 1:
Texto: Busca ingeniería genética en Google. Y después me dices, si son inventos míos.
Etiquetas: {'ZSL': 'Inapropiado', 'FSL': 'Inapropiado', 'CoT': 'Inapropiado'}
------------------------------------------------------------
Comentario 2:
Texto: Porque en el caso de blanca nieves, la historia dice "una niña blanca, como la nieve" describiendo a la protagonista.

Ahora, una reinterpretacion, o adaptación, se puede modificar lo que quiera, pero por lo menos tener la decencia decencia de también reinterpretar el titulo de la producción.

En fin, sin películas y lo que sea, hay otras cosas por las que frustrarse, ejonarese,